In [1]:
from IPython.display import HTML

HTML(
    '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">'
)

# <i class="fas fa-user-shield" style="color:#18ab4b"></i> Linear fraud classifier: SageMaker local mode

Train a <i class="fas fa-bolt" style="color:#6c63ff"></i> **linear SVM** (hinge loss, balanced classes) on synthetic credit-card transactions with the SageMaker SDK v3 `ModelTrainer` in local mode (`Mode.LOCAL_CONTAINER`, a real Docker training job in a bring-your-own-container), evaluate precision/recall, and store the <i class="fas fa-brain" style="color:#18ab4b"></i> model in oblako's <i class="fas fa-database" style="color:#18ab4b"></i> **S3Proxy**.

Same model the original managed-LinearLearner notebook used (`loss=hinge_loss`, `positive_example_weight_mult=balanced`); here it runs anywhere, with no AWS account.

Prereqs: `oblako up s3`, Docker, `pip install 'oblako[sagemaker,notebook]'`.

In [2]:
import logging

# the SageMaker SDK logs every step at INFO through a root rich handler, unless the
# root logger is configured first: keep its output (and botocore's) to warnings
logging.basicConfig(level=logging.WARNING)
logging.getLogger("sagemaker.config").setLevel(logging.WARNING)
# a local endpoint is polled on /ping while it starts; the retries aren't news
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

## <i class="fas fa-file-code" style="color:#6c63ff"></i> Build the container + <i class="fas fa-user-graduate" style="color:#18ab4b"></i> train

In [3]:
import os
import shutil
from pathlib import Path

import frauddata
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, InputData
from sagemaker.train.model_trainer import Mode

from oblako.engines.sagemaker import use_local_stubs
from oblako.services import SageMakerService

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
use_local_stubs()  # no AWS account: skip the SDK's IAM role check

IMAGE = "oblako-fraud:latest"
HP = {"epochs": "30"}

sm = SageMakerService()
sm.build_image(path="container", tag=IMAGE)  # linear-SVM BYOC (./container)

# ModelTrainer's local mode bind-mounts its working dir into the container, so keep
# it under the project: Docker Desktop shares /Users, not the /var/folders temp dir.
work = Path(".sm-local/02").resolve()
shutil.rmtree(work, ignore_errors=True)
(work / "train").mkdir(parents=True)
(work / "job").mkdir()

# Synthetic creditcard.csv-shaped data: Time, V1..V28, Amount, Class (1 = fraud).
train_csv = work / "train" / "train.csv"
frauddata.write_training_csv(str(train_csv), n=4000, seed=0)
test_csv = work / "test.csv"
frauddata.write_training_csv(str(test_csv), n=1500, seed=7)
print(f"train fraud rate: {frauddata.fraud_rate(str(train_csv)) * 100:.1f}%")

trainer = ModelTrainer(
    training_image=IMAGE,
    training_mode=Mode.LOCAL_CONTAINER,
    compute=Compute(instance_type="local", instance_count=1),
    role="arn:aws:iam::000000000000:role/dummy",
    hyperparameters=HP,
    local_container_root=str(work / "job"),
)
trainer.train(
    input_data_config=[InputData(channel_name="train", data_source=str(work / "train"))]
)
model_tar = work / "job" / "compressed_artifacts" / "model.tar.gz"

Step 1/6 : FROM python:3.11-slim
 ---> 1042b61448fe
Step 2/6 : RUN pip install --no-cache-dir scikit-learn pandas joblib
 ---> Using cache
 ---> 2186bd9c3278
Step 3/6 : COPY src/ /opt/program/
 ---> Using cache
 ---> ea3fa526f20c
Step 4/6 : WORKDIR /opt/program
 ---> Using cache
 ---> 96b5e1f0f38d
Step 5/6 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
 ---> 1d76143076b6
Step 6/6 : ENTRYPOINT ["python", "main.py"]
 ---> Using cache
 ---> cdef09d3e53b
Successfully built cdef09d3e53b
Successfully tagged oblako-fraud:latest
train fraud rate: 19.7%


time="2026-09-30T18:57:57+02:00" level=warning msg="a network with name sagemaker-local exists but was not created for project \"job\".\nSet `external: true` to use an existing network"
 Container job-algo-1-1 Recreate 
 Container job-algo-1-1 Recreated 
Attaching to algo-1-1
 Container job-algo-1-1 Starting 


 Container job-algo-1-1 Started 


algo-1-1  | /usr/local/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
algo-1-1  |   warnings.warn(
algo-1-1  | trained on 4000 rows, 19.7% fraud, 30 epochs
algo-1-1  | saved model to /opt/ml/model
algo-1-1 exited with code 0
 Compose Stopping Aborting on container exit...
 Container job-algo-1-1 Stopping 
 Container job-algo-1-1 Stopped 


## <i class="fas fa-chart-line" style="color:#18ab4b"></i> Evaluate on held-out transactions

Unpack the `model.tar.gz` the training job produced and score the test set. Fraud is rare, so precision/recall/F1 matter more than accuracy.

In [4]:
import json
import tarfile
import tempfile

import joblib
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

unpacked = tempfile.mkdtemp(prefix="fraud-model-")
with tarfile.open(model_tar) as tar:
    tar.extractall(unpacked, filter="data")

model = joblib.load(os.path.join(unpacked, "fraud_model.joblib"))
features = json.load(open(os.path.join(unpacked, "model_metadata.json")))[
    "feature_names"
]

test = pd.read_csv(test_csv)
y_true = test["Class"].astype(int)
y_pred = model.predict(test[features])

print(f"fraud in test: {y_true.mean() * 100:.1f}%")
print(f"precision:     {precision_score(y_true, y_pred):.3f}")
print(f"recall:        {recall_score(y_true, y_pred):.3f}")
print(f"f1:            {f1_score(y_true, y_pred):.3f}")
print(f"accuracy:      {accuracy_score(y_true, y_pred):.3f}")

fraud in test: 20.8%
precision:     0.547
recall:        0.846
f1:            0.664
accuracy:      0.822


## <i class="fas fa-database" style="color:#18ab4b"></i> Store the trained model in oblako's S3

In [5]:
from oblako.services import S3ProxyService

s3 = S3ProxyService().get_client()
try:
    s3.create_bucket(Bucket="fraud-models")
except Exception:
    pass
s3.upload_file(str(model_tar), "fraud-models", "fraud/model.tar.gz")
print("stored: s3://fraud-models/fraud/model.tar.gz")
print(
    "bucket:",
    [o["Key"] for o in s3.list_objects_v2(Bucket="fraud-models").get("Contents", [])],
)

stored: s3://fraud-models/fraud/model.tar.gz
bucket: ['fraud/model.tar.gz']
